In [1]:
from pathlib import Path
import pandas as pd

# Configuração
BASE_DIR = Path('/home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research')
ABORDAGEM_DIR = BASE_DIR / 'data' / 'experimentos' / 'abordagem1'
CHAVES = ['V010100', 'NUM_QUADRA', 'NUM_FACE', 'V010800']

# Lista todos os arquivos parquet na pasta
parquet_files = sorted(ABORDAGEM_DIR.glob('*.parquet'))
print(f"Encontrados {len(parquet_files)} arquivos parquet em {ABORDAGEM_DIR}")

# Separa em pares: original (_new) e antigo (sem _new)
pairs = []
for f in parquet_files:
    nome = f.stem
    if '_new' in nome:
        original_name = nome.replace('_new', '')
        original_file = f.with_name(original_name + f.suffix)
        if original_file.exists():
            pairs.append((original_file, f))
        else:
            print(f"Aviso: arquivo original não encontrado para {f.name}")

print(f"\nFormados {len(pairs)} pares para comparação")

# Realiza a comparação para cada par
for original_file, new_file in pairs:
    print(f"\n{'='*60}")
    print(f"Comparando: {original_file.name} vs {new_file.name}")
    print(f"{'='*60}")

    df_original = pd.read_parquet(original_file)
    df_new = pd.read_parquet(new_file)

    print(f"Original -> linhas: {len(df_original)}, colunas: {len(df_original.columns)}")
    print(f"New      -> linhas: {len(df_new)}, colunas: {len(df_new.columns)}")
    print(f"Colunas apenas no original: {set(df_original.columns) - set(df_new.columns)}")
    print(f"Colunas apenas no new: {set(df_new.columns) - set(df_original.columns)}")

    # Verifica se as colunas chaves existem em ambos
    missing_original = [c for c in CHAVES if c not in df_original.columns]
    missing_new = [c for c in CHAVES if c not in df_new.columns]

    if missing_original or missing_new:
        print(f"Colunas chaves ausentes no original: {missing_original}")
        print(f"Colunas chaves ausentes no new: {missing_new}")
        continue

    # Subconjunto com as colunas chaves
    orig_keys = df_original[CHAVES].drop_duplicates()
    new_keys = df_new[CHAVES].drop_duplicates()

    # Converte para tuplas para facilitar comparação
    orig_set = set(map(tuple, orig_keys.values))
    new_set = set(map(tuple, new_keys.values))

    only_in_original = orig_set - new_set
    only_in_new = new_set - orig_set
    in_both = orig_set & new_set

    print(f"Registros únicos por chave no original: {len(orig_set)}")
    print(f"Registros únicos por chave no new: {len(new_set)}")
    print(f"Registros em ambos: {len(in_both)}")
    print(f"Apenas no original: {len(only_in_original)}")
    print(f"Apenas no new: {len(only_in_new)}")


Encontrados 6 arquivos parquet em /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data/experimentos/abordagem1

Formados 3 pares para comparação

Comparando: df_estabel_final.parquet vs df_estabel_final_new.parquet
Original -> linhas: 100000, colunas: 326
New      -> linhas: 100000, colunas: 326
Colunas apenas no original: set()
Colunas apenas no new: set()
Registros únicos por chave no original: 100000
Registros únicos por chave no new: 100000
Registros em ambos: 0
Apenas no original: 100000
Apenas no new: 100000

Comparando: df_lav_temp_final.parquet vs df_lav_temp_final_new.parquet
Original -> linhas: 153876, colunas: 341
New      -> linhas: 153876, colunas: 340
Colunas apenas no original: {'id'}
Colunas apenas no new: set()
Registros únicos por chave no original: 100000
Registros únicos por chave no new: 100000
Registros em ambos: 0
Apenas no original: 100000
Apenas no new: 100000

Comparando: df_pecu_final.parquet vs df_pecu_final_new.parquet
Original -> linhas: 91382,